# Chapter 1 — Defining Agentic AI

Companion notebook to the three Ch1 skills in `skills/crisis/`:
`agent-constraint-triangle-scorer`, `workflow-agent-spectrum-classifier`, and
`vector-vs-graph-retrieval-selector`.

**Running scenario (threaded through the whole book).** A DevOps agent is paged:
the **checkout API is slow** in fictional AWS account `123456789012`. We use that
one incident to make Ch1's abstractions concrete. Every AWS call goes through
[`moto`](https://github.com/getmoto/moto)'s `@mock_aws`, so the notebook runs with
real boto3 shapes and **zero credentials**.

**The arc of this notebook mirrors the arc of the chapter:**

1. The crisis — why naive vector-RAG agents stall in the enterprise.
2. What *agency* actually is — the three dimensions (autonomy / action / authority)
   and the workflow-agent spectrum.
3. The agent constraint triangle — the three coupled constraints that make agents
   hard to build.
4. Why vector retrieval fails the agent — and where GraphRAG and hybrid win.

Each code cell is preceded by the chapter idea it exercises. Read the markdown
first; the code is the idea made runnable.

## 1. The crisis: agents need understanding, not just data

Ch1 opens with a promise — an AI that "doesn't just wait for your questions" but
"pursues your goals" — and then the trap. Wire an LLM to a vector-RAG system and
you inherit **five fatal flaws**:

| Fatal flaw | What the agent experiences |
|---|---|
| Context amnesia | Every conversation starts from scratch; no memory of past incidents. |
| Relationship blindness | Sees the customer and the purchase, not how they connect. |
| Temporal ignorance | Treats an outdated config as the current truth. |
| Reasoning paralysis | Finds *similar text*, not *logical connections*. |
| Tool chaos | Guesses which API to call instead of orchestrating. |

The chapter's thesis: these are **not five separate bugs**. They are symptoms of a
single representational gap — strings instead of *things* (Singhal, Google 2012).
Google closed that gap with a knowledge graph; LinkedIn's KG-augmented RAG cut
median per-issue resolution time by **28.6%** over a six-month deployment.

The rest of Ch1 — and this notebook — builds the vocabulary to diagnose that gap
precisely. We start by defining agency itself.

## 2. Setup — mock AWS and load the three Ch1 skills

Each skill ships a pure-Python `lib.py`. Because all three files are named
`lib.py`, a plain `import lib` (the single-skill pattern from `spike-a`) would
collide. We load each one under its own alias with `importlib` — the same
`sys.path`/file-location idea, generalized to three skills.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path

# Fictional account — moto accepts any 12-digit id. No real credentials used.
FICTIONAL_ACCOUNT_ID = "123456789012"
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "testing")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "testing")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRISIS = PROJECT_ROOT / "skills" / "crisis"


def load_skill(slug: str, alias: str):
    """Load skills/crisis/<slug>/lib.py under a distinct module alias."""
    path = CRISIS / slug / "lib.py"
    sys.path.insert(0, str(path.parent))  # so any intra-skill imports resolve
    spec = importlib.util.spec_from_file_location(alias, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


triangle = load_skill("agent-constraint-triangle-scorer", "triangle_lib")
spectrum = load_skill("workflow-agent-spectrum-classifier", "spectrum_lib")
retrieval = load_skill("vector-vs-graph-retrieval-selector", "retrieval_lib")
print("Loaded 3 Ch1 skills:")
print("  triangle  ->", triangle.__name__)
print("  spectrum  ->", spectrum.__name__)
print("  retrieval ->", retrieval.__name__)

## 3. What is an agent, anyway? The three dimensions of agency

Most definitions jump to implementation (Simon Willison: "an LLM agent runs tools
in a loop to achieve a goal"). Ch1 steps back and defines **agency** along three
dimensions that "exist on sliding scales, not as binary attributes":

- **Autonomy** — how much the system decides on its own, without external direction.
- **Action** — its ability to *effect change* in the environment. This one is a
  gate: "Without this capability to effect change, you have an assistant or
  advisor, **not an agent**."
- **Authority** — the scope and limits of permitted actions (Ch1's real-estate
  agent: free to market a property, no authority over price).

An agent's **environment** is "where the agent performs its tasks and generates
its output." For our DevOps agent that environment includes the EC2 hosts running
the checkout API. Let's stand that environment up under `moto` so "action affects
the environment" is literal, not metaphorical.

In [ ]:
import boto3
from moto import mock_aws


@mock_aws
def provision_checkout_environment(n_hosts: int = 3):
    """The environment the DevOps agent acts on: EC2 hosts for the checkout API."""
    ec2 = boto3.client("ec2")
    ec2.run_instances(
        ImageId="ami-0checkout00000000",
        MinCount=n_hosts,
        MaxCount=n_hosts,
        TagSpecifications=[{
            "ResourceType": "instance",
            "Tags": [{"Key": "service", "Value": "checkout-api"}],
        }],
    )
    described = ec2.describe_instances(
        Filters=[{"Name": "tag:service", "Values": ["checkout-api"]}]
    )
    host_ids = [i["InstanceId"] for r in described["Reservations"] for i in r["Instances"]]
    return host_ids


checkout_hosts = provision_checkout_environment()
print(f"checkout-api environment (account {FICTIONAL_ACCOUNT_ID}):")
for h in checkout_hosts:
    print(f"  host {h}")

### Placing DevOps systems on the workflow-agent spectrum

Ch1 rejects "is it an agent or not?" as the wrong question. Following Andrew Ng
("agent-like to different degrees") and Anthropic's workflow-vs-agent distinction,
systems sit on a **continuous spectrum**:

- **Workflow** end — predefined execution paths (a scripted latency runbook).
- **Blended** middle — deterministic workflow with LLM steps at key points, humans
  in the loop where judgment is required.
- **Agent** end — the system determines its process dynamically.

We score four DevOps systems with `workflow-agent-spectrum-classifier`. Watch the
`latency-advisor`: it reasons with high autonomy but **cannot act**, so despite a
high spectrum position it **fails the action test** — an advisor, not an agent.
That is Ch1's action gate doing its job.

In [ ]:
devops_systems = [
    dict(name="scripted-latency-runbook", autonomy=0.1, action=0.2, authority=0.2,
         path_determinism=0.95, memory=False, tool_use=True, contextual=False),
    dict(name="human-in-loop-remediation", autonomy=0.5, action=0.6, authority=0.4,
         path_determinism=0.55, memory=True, tool_use=True, contextual=True),
    dict(name="autonomous-devops-agent", autonomy=0.9, action=0.8, authority=0.5,
         path_determinism=0.15, memory=True, tool_use=True, contextual=True),
    dict(name="latency-advisor (read-only)", autonomy=0.85, action=0.05, authority=0.05,
         path_determinism=0.2, memory=True, tool_use=True, contextual=True),
]

placements = {}
for s in devops_systems:
    r = spectrum.classify(**s)
    placements[s["name"]] = r
    gate = "AGENT" if r["is_agent_by_action_test"] else "advisor/assistant"
    print(f"{s['name']:<32} pos={r['spectrum_position']:<6} band={r['band']:<9} action-test -> {gate}")
    for note in r["notes"]:
        print(f"      note: {note}")

**Verification gate.** The scripted runbook must land at the WORKFLOW end, the
autonomous agent at the AGENT end, and the read-only advisor must *fail* the action
test regardless of how high it reasons.

In [ ]:
assert placements["scripted-latency-runbook"]["band"] == "WORKFLOW"
assert placements["autonomous-devops-agent"]["band"] == "AGENT"
assert placements["human-in-loop-remediation"]["band"] == "BLENDED"
assert placements["latency-advisor (read-only)"]["is_agent_by_action_test"] is False, \
    "A read-only advisor must fail Ch1's action test."
print("PASS: the three dimensions place each DevOps system correctly, and the")
print("      action test separates the advisor from the true agent.")

## 4. The agent constraint triangle

Once a system is a real agent, Ch1 warns of a "fundamental challenge": the **agent
constraint triangle** — three *interconnected* constraints:

1. **Complexity management** — multistep reasoning; cognitive load "increases
   exponentially" with step count, producing "compounding errors."
2. **Tool orchestration** — translating NL into structured API calls. The failure
   is ambiguity: "If a human engineer can't definitively say which tool should be
   used in a given situation, an AI agent can't be expected to do better."
3. **Context utilization** — the fixed context window is "the model's attention
   budget"; *context rot* means recall degrades as tokens grow (a gradient, not a
   cliff).

They form **competing trade-offs** — pushing on one loads the others:
`complexity -> tools -> context`, `tools -> context -> complexity`,
`context -> complexity -> tools`. The governing principle is *minimal-but-
sufficient* on every axis.

To ground the tool count, we read the sibling 30-tool AWS registry that the
`rag-mcp-tool-selection` skill ships — a realistic curated tool set — and contrast
a **balanced** agent against the Block/Goose failure shape: a **bloated** agent
with every MCP server enabled just in case.

In [ ]:
import json

registry_path = (PROJECT_ROOT / "skills" / "tool-orchestration"
                 / "rag-mcp-tool-selection" / "sample-aws-tools.json")
curated_tool_count = len(json.loads(registry_path.read_text(encoding="utf-8"))["tools"])
print(f"Curated DevOps tool registry: {curated_tool_count} tools\n")

balanced = dict(name="balanced-graph-agent", avg_task_steps=5,
                tool_count=curated_tool_count, tools_disambiguable=True,
                context_window_tokens=200_000, avg_context_tokens_used=60_000)
bloated = dict(name="bloated-mcp-agent", avg_task_steps=8,
               tool_count=curated_tool_count * 3, tools_disambiguable=False,
               context_window_tokens=200_000, avg_context_tokens_used=178_000)

for cfg in (balanced, bloated):
    r = triangle.score(cfg)
    print(f"### {cfg['name']}: {r['overall_band']} (dominant: {r['dominant_constraint']})")
    for name, p in r["pressures"].items():
        print(f"    {name:<24} pressure={p['pressure']:<6} band={p['band']}")
    for c in r["active_pressure_cycles"]:
        print(f"    active cycle: {c['edge']}")
    print()

**Verification gate.** The bloated agent should be OVERCONSTRAINED with
tool-orchestration as its dominant corner (ambiguity + catalog size) and at least
one active pressure cycle; the balanced agent should read BALANCED.

In [ ]:
b = triangle.score(bloated)
g = triangle.score(balanced)
assert b["overall_band"] == "OVERCONSTRAINED"
assert b["dominant_constraint"] == "tool_orchestration"
assert b["active_pressure_cycles"], "an overconstrained agent must fire a pressure cycle"
assert g["overall_band"] == "BALANCED"
print("PASS: the constraint triangle flags the bloated agent as OVERCONSTRAINED")
print(f"      (dominant corner: {b['dominant_constraint']}) and the balanced agent as BALANCED.")
print(f"      Fix suggested: {b['recommendations'][0]['action'][:88]}...")

## 5. Why vector retrieval fails the agent — and where GraphRAG wins

Retrieval is the failure point everything downstream amplifies: "When retrieval fails, the rest of the
system fails." Ch1 quantifies exactly where vector RAG works, using Microsoft's
BenchmarkQED (scope: local/global × type: data/activity):

- **DataLocal** simple lookups: vector RAG ~**90%**.
- **ActivityGlobal** complex reasoning: vector RAG **20-30%**.
- **LazyGraphRAG** beats vector RAG by **50-60%** on multi-hop reasoning.
- At **100,000 pages**, vector accuracy drops up to **12%**; graph drops only ~**2%**.

Our DevOps incident produces both kinds of query. Let's first *demonstrate the
DataLocal case where vector RAG genuinely shines* — a specific 5xx lookup — against
mocked CloudWatch Logs (the proven `spike-a` seam), then classify three DevOps
workloads with `vector-vs-graph-retrieval-selector`.

In [ ]:
import time


@mock_aws
def datalocal_5xx_lookup():
    """A DataLocal query: 'show the 5xx errors in the checkout log'. This is the
    case vector RAG handles at ~90% — the answer lives in a specific region."""
    logs = boto3.client("logs")
    lg = "/aws/ecs/checkout-api"
    logs.create_log_group(logGroupName=lg)
    logs.create_log_stream(logGroupName=lg, logStreamName="task-abc123")
    now_ms = int(time.time() * 1000)
    logs.put_log_events(logGroupName=lg, logStreamName="task-abc123", logEvents=[
        {"timestamp": now_ms - 180_000, "message": "ERROR checkout req-013 status=504 db_pool_exhausted=true"},
        {"timestamp": now_ms - 60_000,  "message": "ERROR checkout req-019 status=504 db_pool_exhausted=true"},
    ])
    started = logs.start_query(
        logGroupName=lg,
        startTime=int(now_ms / 1000) - 3600,
        endTime=int(now_ms / 1000),
        queryString="fields @timestamp,@message | filter status=504 | sort @timestamp desc",
    )
    return started["queryId"]


query_id = datalocal_5xx_lookup()
print(f"DataLocal 5xx lookup issued against mocked CloudWatch Logs -> queryId={query_id}")
print("This is exactly the local, fact-based lookup vector RAG is great at.")

Now the harder queries. The DevOps agent also asks things vector RAG **cannot**
answer — the chapter's **associativity gap** example: *"which services were affected
by the database migration that followed the security patch we discussed last
month?"* That requires traversing relationships across documents and time; there is
no single chunk to retrieve. And an autonomous agent constantly moves between local
and global understanding, which is why Ch1 recommends a **hybrid** architecture.

In [ ]:
workloads = [
    dict(name="specific-5xx-lookup", query_scope="local", query_type="data",
         multi_hop=False, temporal=False, structured_domain=False, latency_critical=True),
    dict(name="cascading-migration-impact", query_scope="global", query_type="activity",
         multi_hop=True, temporal=True, structured_domain=True),
    dict(name="autonomous-devops-agent", query_scope="mixed", query_type="activity",
         multi_hop=True, temporal=True, structured_domain=True, agentic=True,
         dataset_scale_pages=120_000),
]

recs = {}
for w in workloads:
    r = retrieval.recommend(**w)
    recs[w["name"]] = r
    print(f"{w['name']:<28} -> {r['recommendation']}")
    print(f"      {r['reasons'][0]}")
print()

# The larger-context-window rebuttal, on demand.
proposal = retrieval.recommend(query_scope="global", query_type="activity",
                               multi_hop=True, larger_context_window=True)
print("Someone proposes: 'just use a 1M-token context window instead of a graph.'")
print("Rebuttal:", proposal["larger_context_window_rebuttal"][:220], "...")

**Verification gate.** The local 5xx lookup routes to VECTOR (where it belongs);
the cross-document, temporal migration query routes to GRAPH; the autonomous agent
routes to HYBRID (Ch1's recommended default); and the bigger-window proposal returns
the BenchmarkQED 1M-token rebuttal.

In [ ]:
assert recs["specific-5xx-lookup"]["recommendation"] == "VECTOR"
assert recs["cascading-migration-impact"]["recommendation"] == "GRAPH"
assert recs["autonomous-devops-agent"]["recommendation"] == "HYBRID"
assert isinstance(query_id, str) and query_id, "moto CloudWatch Logs start_query must return a queryId"
assert "1-million-token" in proposal["larger_context_window_rebuttal"]
print("PASS: local lookup -> VECTOR, associativity query -> GRAPH, agent -> HYBRID,")
print("      and the larger-context-window rebuttal cites the 1M-token BenchmarkQED test.")

## 6. Verification battery and verdict

One consolidated battery over all three Ch1 skills, then the chapter verdict.

In [ ]:
checks = []

# Spectrum: action test separates advisor from agent.
checks.append(("action-test-separates-advisor",
    spectrum.classify(0.85, 0.05, 0.05, 0.2)["is_agent_by_action_test"] is False))

# Triangle: ambiguous tools raise tool-orchestration pressure above a clean set.
checks.append(("ambiguity-dominates-tool-pressure",
    triangle.score_tool_orchestration(30, False) > triangle.score_tool_orchestration(30, True)))

# Triangle: context pressure follows the rot gradient (rises with fill).
checks.append(("context-rot-gradient",
    triangle.score_context_utilization(200_000, 40_000)
    < triangle.score_context_utilization(200_000, 180_000)))

# Retrieval: DataLocal -> VECTOR, ActivityGlobal multi-hop -> GRAPH.
checks.append(("datalocal-is-vector",
    retrieval.recommend(query_scope="local", query_type="data",
                        latency_critical=True, structured_domain=False)["recommendation"] == "VECTOR"))
checks.append(("activityglobal-multihop-is-graph",
    retrieval.recommend(query_scope="global", query_type="activity",
                        multi_hop=True, structured_domain=True)["recommendation"] == "GRAPH"))

# Retrieval: the ActivityGlobal quadrant carries the 20-30% vector anchor.
checks.append(("benchmarkqed-20-30-anchor",
    "20-30%" in retrieval.BENCHMARKQED_QUADRANTS["activity_global"]["vector_accuracy"]))

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in checks), "Ch1 verification battery failed."
print(f"\nAll {len(checks)} Ch1 gates passed.")

## Verdict

We walked Chapter 1's argument end to end on one DevOps incident:

| Gate | Status |
|------|--------|
| Three Ch1 skills load under distinct aliases | PASS |
| Agency dimensions place four DevOps systems; action test separates advisor from agent | PASS |
| Constraint triangle flags the bloated agent OVERCONSTRAINED, the balanced one BALANCED | PASS |
| Retrieval selector routes local->VECTOR, associativity->GRAPH, agent->HYBRID | PASS |
| moto-mocked CloudWatch Logs + EC2 return shaped data (zero credentials) | PASS |
| Larger-context-window rebuttal cites the 1M-token BenchmarkQED test | PASS |

**The through-line.** An *agent* (not a workflow, not an advisor) pursues goals
across autonomy / action / authority, but it is boxed in by the constraint triangle
and — most decisively — by retrieval. Vector RAG is excellent for the DataLocal 5xx
lookup and useless for the cross-service, temporal, multi-hop questions a real
incident demands. That is the representational gap Ch1 diagnoses, and it is why the
rest of the book builds toward GraphRAG and the hybrid architecture. **Chapter 2**
introduces the dual-graph architecture and the eight pillars that close the gap.